# 1 · Data and measures

This page builds the shared analysis data from the raw study records.
The inputs have one row per child and one row per directed decision:
child $i$ sends tokens to classmate $j$.

The preparation keeps the original fields and adds paired transfers,
friendship type, Social Rank, the two gaps, and the quantities needed
for the null-model filter. The student file adds `pagerank`,
`social_rank`, and `attendance_z`; Figure 1D uses attendance
standardized within classroom and GPA on its original scale.

# What each step builds

Run this page from top to bottom to build the data. The steps add
columns to the same two working tables, `dyads` and `students`. Tables
such as the reverse decisions, popularity edges, and simulated transfers
are intermediate objects in R. They are used in the next calculation;
they are not saved as separate datasets.

| Step | Working table | Columns added |
|------------------------|------------------------|------------------------|
| 1\. Match reverse decisions | `dyads` | `tokens_received` |
| 2\. Cooperation GAP | `dyads` | `coop_gap` |
| 3\. Friendship type | `dyads` | `friendship` |
| 4\. Social Rank | `students` | `pagerank`, `social_rank` |
| 5\. Social Rank GAP | `dyads` | `sender_rank`, `receiver_rank`, `rank_gap` |
| 6\. Balance covariates | `students` | `attendance_z` |
| 7\. Null-model filter | `dyads` | Null means, SDs, departure scores, filtered transfers, and `coop_gap_filtered` |

Only the final **Save the prepared data** step writes files:
`analysis_dyads.csv` and `analysis_students.csv` in
`data/generated/main/`. The table and figure pages then reuse these
objects in R or read their saved CSV files. If those inputs are missing,
a result page runs this preparation automatically.

In [ ]:
here::i_am("notebooks/main/01_data_and_measures.qmd")
library(tidyverse)
library(igraph)
library(here)

# Inputs and units

Download the [data and code from GitHub
(ZIP)](https://github.com/melanieoyarzun/friendship-hierarchy/archive/refs/heads/main.zip).
The input files are in `data/raw/`:
[students.csv](https://raw.githubusercontent.com/melanieoyarzun/friendship-hierarchy/main/data/raw/students.csv),
[dyads.csv](https://raw.githubusercontent.com/melanieoyarzun/friendship-hierarchy/main/data/raw/dyads.csv),
and
[null_model_order.csv](https://raw.githubusercontent.com/melanieoyarzun/friendship-hierarchy/main/data/raw/null_model_order.csv).
The [raw-data
dictionary](https://github.com/melanieoyarzun/friendship-hierarchy/blob/main/data/raw/codebook.md)
describes their fields.

The game involved 859 children in 45 classrooms. Each child allocated
0–10 tokens to each classmate. For a pair, the dataset contains both
$i \to j$ and $j \to i$. `tokens_sent` and `tokens_received` record the
original allocations before the game’s doubling of received tokens.

In [ ]:
dyads    <- read_csv(here("data", "raw", "dyads.csv"),    show_col_types = FALSE)
students <- read_csv(here("data", "raw", "students.csv"), show_col_types = FALSE)

In [ ]:
tibble(children = nrow(students), classrooms = n_distinct(dyads$class_id),
       directed_decisions = nrow(dyads))

children,classrooms,directed_decisions
859,45,17804


In [ ]:
dyads %>% select(class_id, round, sender_id, receiver_id, tokens_sent) %>% head()

class_id,round,sender_id,receiver_id,tokens_sent
1,1,30,328,1
1,1,68,230,4
1,1,125,161,3
1,1,156,319,5
1,1,161,125,2
1,1,224,711,1


# Step 1. Match the partner’s transfer

`tokens_received` is the partner’s transfer in the reverse decision.
Both decisions of a pair occur in the same classroom and round. Matching
the two directions preserves all 17,804 directed rows.

In [ ]:
# Each row is one decision: sender i gives `tokens_sent` (0 to 10) to receiver j.
# In the same round, j also decided how much to give to i. That reverse
# decision is what i receives. We find it by swapping sender and receiver.

reverse_decisions <- dyads %>%
  select(class_id,
         sender_id   = receiver_id,
         receiver_id = sender_id,
         tokens_received = tokens_sent,
         round_reverse = round)

dyads <- dyads %>%
  left_join(reverse_decisions, by = c("class_id", "sender_id", "receiver_id"))

# every decision has its reverse, and both happened in the same round
stopifnot(!anyNA(dyads$tokens_received))
stopifnot(all(dyads$round == dyads$round_reverse))

dyads <- dyads %>% select(-round_reverse)

In [ ]:
dyads %>% select(sender_id, receiver_id, tokens_sent, tokens_received) %>% head()

sender_id,receiver_id,tokens_sent,tokens_received
30,328,1,10
68,230,4,1
125,161,3,2
156,319,5,2
161,125,2,3
224,711,1,4


# Step 2. Cooperation GAP

`coop_gap = tokens_received - tokens_sent`, in tokens, with range −10 to
10. A positive value means the focal child receives more than she sends.
This is a difference in allocations, with no normalization or payoff
multiplier. The reverse decision has the opposite gap.

In [ ]:
# Cooperation GAP (from the sender's point of view): what i receives minus what
# i sends. Positive values mean i receives more than i gives.

dyads <- dyads %>%
  mutate(coop_gap = tokens_received - tokens_sent)

In [ ]:
summary(dyads$coop_gap)
##    Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
##     -10      -3       0       0       3      10

# Step 3. Friendship type

Question 3 asked for best friends. `friendship` is `Mutual` if both
children named each other, `Non-friends` if neither did, and one-sided
if only one did. The two one-sided directions are `Sender declares` and
`Receiver declares`.

In [ ]:
# Question 3: "Select 2-5 classmates you consider your best friends".
# nom_q3 = 1 if the sender named the receiver. To know if the receiver also
# named the sender, we look at the reverse row, as we did with the tokens.

reverse_friend <- dyads %>%
  select(class_id,
         sender_id   = receiver_id,
         receiver_id = sender_id,
         friend_reverse = nom_q3)

dyads <- dyads %>%
  left_join(reverse_friend, by = c("class_id", "sender_id", "receiver_id")) %>%
  mutate(friendship = case_when(
    nom_q3 == 1 & friend_reverse == 1 ~ "Mutual",
    nom_q3 == 1 & friend_reverse == 0 ~ "Sender declares",    # one-sided, i named j
    nom_q3 == 0 & friend_reverse == 1 ~ "Receiver declares",  # one-sided, j named i
    nom_q3 == 0 & friend_reverse == 0 ~ "Non-friends"
  ))

cat("Decisions by friendship type:\n")
dyads %>% count(friendship) %>% mutate(percent = round(100 * n / sum(n), 2))

In [ ]:
dyads %>% count(friendship) %>% mutate(percent = 100 * n / sum(n))

friendship,n,percent
Mutual,1288,7.234329
Non-friends,13916,78.162211
Receiver declares,1300,7.301730
Sender declares,1300,7.301730


# Step 4. Social Rank

Within each classroom, the popularity network combines positive
nominations (Q4) and negative nominations (Q5). The edge from $i$ to $j$
has weight $a_{ij}=P^+_{ij}-P^-_{ij}+1$: 2 for a positive nomination
alone, 0 for a negative nomination alone, and 1 when the two answers
cancel or neither nomination is present.

In [ ]:
# Questions 4 and 5: "most popular and well-known" and "not popular or
# well-known". For every ordered pair (i, j) in a classroom:
#
#     s_ij = nom_q4 - nom_q5      +1 popular, -1 not popular, 0 not named
#     a_ij = s_ij + 1             weights 2, 0 and 1 (all non-negative)
#
# a_ij is the weight of the edge i -> j in the popularity network.

popularity_edges <- dyads %>%
  transmute(class_id,
            from = as.character(sender_id),
            to   = as.character(receiver_id),
            weight = nom_q4 - nom_q5 + 1)

cat("Edges of the popularity network by weight (2 popular, 0 not popular, 1 not named):\n")
count(popularity_edges, weight)

In [ ]:
popularity_edges %>% count(weight)

weight,n
0,1788
1,13868
2,2148


PageRank uses damping 0.85. It is normalized within each classroom as
`(pagerank - min(pagerank)) / (max(pagerank) - min(pagerank))` to obtain
`social_rank` from 0 to 1.

In [ ]:
# PageRank (damping 0.85) is computed separately in each classroom.
# We then rescale it within each classroom to go from 0 (lowest rank in the
# class) to 1 (highest rank in the class). This is our Social Rank.

pagerank_one_class <- function(edges_one_class) {
  network <- graph_from_data_frame(edges_one_class %>% select(from, to, weight),
                                   directed = TRUE)
  pr <- page_rank(network, damping = 0.85, directed = TRUE)$vector   # uses edge weights
  tibble(student_id = as.numeric(names(pr)), pagerank = pr)
}

social_rank <- popularity_edges %>%
  group_by(class_id) %>%
  group_modify(~ pagerank_one_class(.x)) %>%
  mutate(social_rank = (pagerank - min(pagerank)) / (max(pagerank) - min(pagerank))) %>%
  ungroup()

students <- students %>%
  left_join(social_rank %>% select(student_id, pagerank, social_rank),
            by = "student_id")

cat("Social Rank (0 to 1 within each classroom):\n")
summary(social_rank$social_rank)

In [ ]:
students %>% filter(class_id == 16) %>%
  select(student_id, female, pagerank, social_rank) %>% arrange(desc(social_rank))

student_id,female,pagerank,social_rank
657,0,0.1590302,1.0000000
505,1,0.1282099,0.5930586
546,1,0.1252742,0.5542959
687,1,0.1080575,0.3269715
305,1,0.1056854,0.2956514
257,0,0.1022316,0.2500477
662,0,0.0983122,0.1982968
176,1,0.0899051,0.0872924
808,1,0.0832939,0.0000000


# Step 5. Social Rank GAP

`rank_gap = sender_rank - receiver_rank`, from −1 to 1. Positive values
mean the sender has higher Social Rank. Normalization has already
occurred in `social_rank`; the gap is not standardized again. Rounding
to ten decimal places removes numerical noise around equal ranks.

In [ ]:
# Social Rank GAP of a decision = sender's rank - receiver's rank.
# Positive: the sender has higher rank. It goes from -1 to 1.

dyads <- dyads %>%
  left_join(social_rank %>% select(sender_id = student_id, sender_rank = social_rank),
            by = "sender_id") %>%
  left_join(social_rank %>% select(receiver_id = student_id, receiver_rank = social_rank),
            by = "receiver_id") %>%
  mutate(rank_gap = sender_rank - receiver_rank,
         # round away floating-point noise, so that two children with the
         # same rank have a gap of exactly 0 (and not 1e-16)
         rank_gap = round(rank_gap, 10))

cat("Social Rank GAP (sender - receiver):\n")
summary(dyads$rank_gap)

In [ ]:
summary(dyads$rank_gap)
##    Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
## -1.0000 -0.2211  0.0000  0.0000  0.2211  1.0000

# Step 6. Attendance for the balance check

Figure 1D compares partner rank with the sender’s sex, attendance, and
GPA. Attendance is standardized within classroom as
`(attendance - mean(attendance)) / sd(attendance)`, with missing values
omitted when computing the mean and SD. GPA retains its original 1–7
scale.

In [ ]:
# Figure 1D uses attendance standardized within classroom and GPA in its raw scale.
students <- students %>%
  group_by(class_id) %>%
  mutate(attendance_z = (attendance - mean(attendance, na.rm = TRUE)) /
                         sd(attendance, na.rm = TRUE)) %>%
  ungroup()

In [ ]:
students %>% select(student_id, class_id, female, attendance, attendance_z, gpa) %>% head()

student_id,class_id,female,attendance,attendance_z,gpa
30,1,1,86.02,-0.2203279,6.1
68,1,0,98.92,0.8146928,6.2
125,1,1,80.65,-0.6511853,6.1
133,1,1,94.62,0.4696859,5.5
156,1,0,93.89,0.4111150,5.3
161,1,1,93.55,0.3838354,6.0


# Step 7. Null-model filter

The filter keeps transfers that depart most from the sender’s allocation
pattern under a random reassignment of partners. It starts from all
directed transfers, including zeros.

## 7a. Fix the simulation settings

In [ ]:
n_shuffles <- 1000
null_seed <- 109
retention <- 0.50

For each sender, shuffle her allocations across her classmates 1,000
times, excluding herself. Each shuffle preserves the same amounts, their
total, and the zeros; only their recipients change. The seed is 109.
`data/raw/null_model_order.csv` fixes the sequence of classrooms and
children that consume the random draws. Its ordering fields are
simulation instructions.

## 7b. Score each observed transfer

For each directed transfer, save the mean and SD of its 1,000 simulated
values as `null_mean` and `null_sd`. The departure score is

$$
d_{ij}=\frac{|w_{ij}-\mu^{\mathrm{null}}_{ij}|}{\sigma^{\mathrm{null}}_{ij}}.
$$

When the null SD is zero, set the score to zero. This occurs when a
sender allocates the same amount to every classmate. The score measures
distance from the null mean; it is not a p-value.

In [ ]:
# Null-model filtering of the directed token transfers.
#
# Stage 1. In each classroom we write the token exchanges as a matrix: one row
# per sender, one column per receiver. For each sender we shuffle her row
# 1,000 times: the same allocations (zeros included) go to her classmates in a
# random order, so her total and her full 0-10 profile are kept and only *who*
# receives each amount changes. Each shuffle gives a simulated value to every
# cell i -> j. The 1,000 simulated values of a cell are its null distribution;
# we keep their mean and standard deviation, and score the observed transfer by
# how far it is from that mean, in standard deviations:
#
#     d_ij = | tokens_sent_ij - null_mean_ij | / null_sd_ij     (0 if null_sd_ij = 0)
#
# The draws are random, so the seed is fixed (109). The shuffles are drawn
# classroom by classroom and child by child, in the order given in
# data/raw/null_model_order.csv. The seed and this fixed sequence determine
# the simulation reproducibly.


null_model_order <- read_csv(here("data", "raw", "null_model_order.csv"), show_col_types = FALSE)

simulate_one_class <- function(class_dyads, class_order) {
  # children in the supplied simulation order
  ids <- class_order %>% arrange(null_model_child_order) %>% pull(student_id)

  # matrix of tokens sent: row = sender, column = receiver
  tokens <- matrix(0, length(ids), length(ids))
  tokens[cbind(match(class_dyads$sender_id, ids), match(class_dyads$receiver_id, ids))] <-
    class_dyads$tokens_sent

  map_dfr(seq_along(ids), function(i) {
    others <- seq_along(ids)[-i]          # classmates, without the diagonal
    row    <- tokens[i, others]           # what child i gave to each of them

    # one column per shuffle: row i with its values in a random order
    shuffles <- matrix(replicate(n_shuffles, row[sample.int(length(row))]), nrow = length(row))

    tibble(sender_id   = ids[i],
           receiver_id = ids[others],
           null_mean   = rowMeans(shuffles),        # mean of the simulated values of each cell
           null_sd     = apply(shuffles, 1, sd))    # their standard deviation
  })
}

set.seed(null_seed)
null_model <- null_model_order %>%
  distinct(class_id, null_model_class_order) %>%
  arrange(null_model_class_order) %>%
  pull(class_id) %>%
  map_dfr(function(cl) {
    simulate_one_class(dyads %>% filter(class_id == cl),
                       null_model_order %>% filter(class_id == cl))
  })

dyads <- dyads %>%
  left_join(null_model, by = c("sender_id", "receiver_id")) %>%
  mutate(departure = if_else(null_sd > 0, abs(tokens_sent - null_mean) / null_sd, 0))

cat("Null model: shuffles per sender:", n_shuffles, "| seed:", null_seed, "\n")
cat("Senders who give the same amount to everyone (score 0 for all their transfers):",
    n_distinct(dyads$sender_id[dyads$null_sd == 0]), "\n")

In [ ]:
example_sender <- dyads$sender_id[1]
dyads %>% filter(sender_id == example_sender) %>%
  select(receiver_id, tokens_sent, null_mean, null_sd, departure) %>%
  arrange(desc(departure))

receiver_id,tokens_sent,null_mean,null_sd,departure
293,10,3.542,2.719123,2.3750305
469,10,3.606,2.723266,2.3479157
161,9,3.735,2.760506,1.9072588
598,0,3.855,2.788503,1.3824620
859,0,3.706,2.711626,1.3667075
621,0,3.548,2.602397,1.3633586
68,0,3.690,2.733015,1.3501574
281,0,3.539,2.636177,1.3424744
312,0,3.610,2.764696,1.3057494
328,1,3.628,2.700539,0.9731392


## 7c. Retain transfers and match the two directions

Rank all transfers in the sample by `departure` and take the cutoff at
50% retention. This is one cutoff across the full sample. Keep scores at
or above the cutoff, so ties at the boundary are kept. Excluded
transfers become `NA`: a retained allocation of zero remains a real
zero.

The reverse direction is filtered using its own departure score. Thus
`tokens_sent_filtered` and `tokens_received_filtered` can be missing
separately. `coop_gap_filtered` is defined only when both transfers are
retained.

In [ ]:
# Stage 2. Keep the 50% of all directed transfers with the largest departure
# (one common cutoff for the whole sample). Dropped transfers become NA, not 0,
# because 0 is a real decision in the game.
# Stage 3. A dyad enters the filtered Cooperation GAP only if BOTH directions
# are kept.

cutoff <- sort(dyads$departure, decreasing = TRUE)[ceiling(retention * nrow(dyads))]

reverse_departure <- dyads %>%
  select(class_id,
         sender_id   = receiver_id,
         receiver_id = sender_id,
         departure_reverse = departure)

dyads <- dyads %>%
  left_join(reverse_departure, by = c("class_id", "sender_id", "receiver_id")) %>%
  mutate(
    tokens_sent_filtered     = if_else(departure >= cutoff,         tokens_sent,     NA_real_),
    tokens_received_filtered = if_else(departure_reverse >= cutoff, tokens_received, NA_real_),
    coop_gap_filtered        = tokens_received_filtered - tokens_sent_filtered
  ) %>%
  select(-departure_reverse)

cat(sprintf("Share of transfers kept:                     %.3f\n",
            mean(!is.na(dyads$tokens_sent_filtered))))   # about 0.50
cat(sprintf("Share of decisions with both directions kept: %.3f\n",
            mean(!is.na(dyads$coop_gap_filtered))))      # about 0.26

In [ ]:
tibble(cutoff = cutoff, directed_decisions = nrow(dyads),
       retained_transfers = sum(!is.na(dyads$tokens_sent_filtered)),
       decisions_with_both_retained = sum(!is.na(dyads$coop_gap_filtered)),
       pairs_with_both_retained = sum(!is.na(dyads$coop_gap_filtered)) / 2)

cutoff,directed_decisions,retained_transfers,decisions_with_both_retained,pairs_with_both_retained
0.7105358,17804,8902,4592,2296


For these data, 8,902 of 17,804 transfers are retained. Both directions
pass for 4,592 directed decisions, representing 2,296 pairs. The
filtered regressions use `coop_gap_filtered`.

# Save the prepared data

The completed working tables become `analysis_dyads` and
`analysis_students` and are saved as the two CSV files below. No earlier
step writes an analysis file. Running this page again rebuilds both
files from the raw records. The saved dyad data include the null means,
SDs, and departure scores, so the filter can be inspected after
reloading.

In [ ]:
dir.create(here("data", "generated", "main"), recursive = TRUE, showWarnings = FALSE)

analysis_dyads <- dyads %>%
  select(class_id, round, sender_id, receiver_id,
         tokens_sent, tokens_received, coop_gap,
         friendship, sender_rank, receiver_rank, rank_gap,
         null_mean, null_sd, departure,
         tokens_sent_filtered, tokens_received_filtered, coop_gap_filtered,
         seconds_since_start, starts_with("nom_q"))
analysis_students <- students

write_csv(analysis_dyads, here("data", "generated", "main", "analysis_dyads.csv"))
write_csv(analysis_students, here("data", "generated", "main", "analysis_students.csv"))
dyads <- analysis_dyads
students <- analysis_students

In [ ]:
tibble(object = c("analysis_dyads", "analysis_students"),
       saved_file = c("analysis_dyads.csv", "analysis_students.csv"),
       rows = c(nrow(analysis_dyads), nrow(analysis_students)),
       columns = c(ncol(analysis_dyads), ncol(analysis_students)))

object,saved_file,rows,columns
analysis_dyads,analysis_dyads.csv,17804,27
analysis_students,analysis_students.csv,859,16


Continue to [Table 1](02_table1_main_models.ipynb), or choose a result
from the [Main Results index](../README.md#main-text). The later notebooks load these
prepared data before fitting a model or constructing a figure.

# Session information

In [ ]:
sessionInfo()
## R version 4.3.2 (2023-10-31)
## Platform: aarch64-apple-darwin20 (64-bit)
## Running under: macOS 26.7.1
## 
## Matrix products: default
## BLAS:   /Library/Frameworks/R.framework/Versions/4.3-arm64/Resources/lib/libRblas.0.dylib 
## LAPACK: /Library/Frameworks/R.framework/Versions/4.3-arm64/Resources/lib/libRlapack.dylib;  LAPACK version 3.11.0
## 
## locale:
## [1] C.UTF-8/C.UTF-8/C.UTF-8/C/C.UTF-8/C.UTF-8
## 
## time zone: America/Santiago
## tzcode source: internal
## 
## attached base packages:
## [1] stats     graphics  grDevices utils     datasets  methods   base     
## 
## other attached packages:
##  [1] igraph_2.1.4    lubridate_1.9.4 forcats_1.0.0   stringr_1.5.1  
##  [5] dplyr_1.1.4     purrr_1.0.4     readr_2.1.5     tidyr_1.3.1    
##  [9] tibble_3.2.1    ggplot2_3.5.2   tidyverse_2.0.0 here_1.0.1     
## 
## loaded via a namespace (and not attached):
##  [1] bit_4.6.0         gtable_0.3.6      jsonlite_2.0.0    crayon_1.5.3     
##  [5] compiler_4.3.2    tidyselect_1.2.1  parallel_4.3.2    scales_1.3.0     
##  [9] yaml_2.3.10       fastmap_1.2.0     R6_2.6.1          generics_0.1.3   
## [13] knitr_1.50        munsell_0.5.1     rprojroot_2.0.4   pillar_1.10.2    
## [17] tzdb_0.5.0        rlang_1.1.6       stringi_1.8.7     xfun_0.52        
## [21] bit64_4.6.0-1     timechange_0.3.0  cli_3.6.4         withr_3.0.2      
## [25] magrittr_2.0.3    digest_0.6.37     grid_4.3.2        vroom_1.6.5      
## [29] hms_1.1.3         lifecycle_1.0.4   vctrs_0.6.5       evaluate_1.0.3   
## [33] glue_1.8.0        colorspace_2.1-1  rmarkdown_2.29    tools_4.3.2      
## [37] pkgconfig_2.0.3   htmltools_0.5.8.1

# Data dictionary

Identifiers are study codes. The following fields cover the raw inputs
and shared analysis data used in the main-result notebooks.

## Raw data

### `students.csv`

##### one row per child (859 rows)

| Variable | Description |
|------------------------------------|------------------------------------|
| `student_id` | Study code of the child |
| `class_id` | Classroom (1–45) |
| `school_id` | School (1–14) |
| `grade` | Grade level (3, 4 or 5, Chilean primary school) |
| `female` | 1 = girl, 0 = boy |
| `age` | Age in years on the day of the experiment (one decimal) |
| `gpa` | Grade point average, Chilean 1–7 scale |
| `attendance` | Attendance in the previous semester (% of school days); used in the balance tests |
| `attendance_2017` | Attendance in the 2017 school year (%); recorded as a separate attendance measure |
| `guardian_secondary` | 1 = guardian completed secondary education, or has any technical or higher education |
| `guardian_education` | Guardian education in four levels: `1 Primary or less` (including no schooling), `2 Secondary incomplete`, `3 Secondary complete`, `4 Tertiary` (any technical-professional or higher education), as in SI Table S9 and Figure S8 |
| `repeated_grade` | 1 = the child repeated a grade, 0 = never; missing when not recorded (249 children) |
| `religion` | Recorded religion: `Catholic` (373), `Evangelical` (341), or `No answer` (145). Source `Católica` and `Evangélica` are translated; missing source values and `N/A` are grouped as `No answer`, as in SI Table S9 and Figure S8. `No answer` does not imply no religious affiliation |

### `dyads.csv`

##### one row per directed decision (17,804 rows)

| Variable | Description |
|------------------------------------|------------------------------------|
| `class_id` | Classroom |
| `round` | Classroom round in which the pair played |
| `sender_id` | Child who decides |
| `receiver_id` | Partner who receives |
| `tokens_sent` | Tokens (0–10) allocated by sender $i$ to partner $j$, denoted $w_{ij}$ in the paper. Recorded before the game multiplier |
| `seconds_since_start` | Seconds between the start of the class session and this decision. Missing in 16 classrooms where the tablets did not record the time of each decision |
| `nom_q1` … `nom_q9` | 1 if the sender named the receiver in nomination question *k* |

### `null_model_order.csv`

##### one row per child (859 rows)

Fixed sequence in which the null-model filter visits classrooms and
children when drawing random shuffles. This sequence and seed 109
determine the simulated values reproducibly. These ordering fields are
used for simulation, not as explanatory variables.

| Variable | Description |
|------------------------------------|------------------------------------|
| `class_id` | Classroom |
| `student_id` | Study code of the child |
| `null_model_class_order` | Position of the classroom in the order of the shuffles (1–45) |
| `null_model_child_order` | Position of the child within her classroom |

Nomination questions (each child named 2 to 5 classmates):

1.  would most like to hang out with
2.  would least like to hang out with
3.  **best friends** (defines friendship type)
4.  **most popular and well-known** (Social Rank, positive)
5.  **not popular or well-known** (Social Rank, negative)
6.  start fights
7.  make fun of others
8.  are kind to others
9.  cooperate

### `figure2_layout.csv`

##### node positions for the assembled Figure 2 (9 rows)

| Variable | Description |
|------------------------------------|------------------------------------|
| `student_id` | Study code of a child in class 16 |
| `x`, `y` | Position of the child in the paper’s drawing of Figure 2 (scaled to −1…1) |

These coordinates reproduce the drawing in Figure 2 and are used only
for plotting.

## Prepared main-result data

The files in `data/generated/main/` retain the raw fields and add the
measures below. Each directed decision remains a row; each child remains
a row in the student file.

| File | Added field | Meaning |
|------------------------|------------------------|------------------------|
| `analysis_students.csv` | `pagerank` | Popularity-network PageRank within classroom, damping 0.85 |
|  | `social_rank` | PageRank rescaled to 0–1 within classroom |
|  | `attendance_z` | Attendance z-score within classroom, used in Figure 1D |
| `analysis_dyads.csv` | `tokens_received` | Partner’s allocation in the reverse decision, 0–10 tokens before the game multiplier |
|  | `coop_gap` | `tokens_received - tokens_sent`, from −10 to 10 tokens |
|  | `friendship` | Mutual, non-friends, sender declares, or receiver declares |
|  | `sender_rank`, `receiver_rank` | Each child’s normalized Social Rank |
|  | `rank_gap` | `sender_rank - receiver_rank`, from −1 to 1, rounded to ten decimals |
|  | `null_mean`, `null_sd` | Mean and SD of the transfer’s 1,000 simulated values |
|  | `departure` | Absolute distance from the null mean divided by null SD; zero when the SD is zero |
|  | `tokens_sent_filtered`, `tokens_received_filtered` | Transfers passing the global 50% departure cutoff in their respective directions; excluded values are missing |
|  | `coop_gap_filtered` | Received minus sent when both directions pass the filter |

The token variables and Cooperation GAP are not normalized. Social Rank
is normalized before forming the rank gap. The filter changes which
transfers contribute to a filtered quantity; it leaves the observed
variables intact.

## Graph edge attributes

Graph constructors use the conventional edge attribute `weight`. In the
popularity graph it is $a_{ij}=P^+_{ij}-P^-_{ij}+1$, with values 0, 1,
or 2. In a game-transfer graph it is a copy of `tokens_sent`. Its
meaning follows the graph being constructed; it does not replace the
token variable in the analysis datasets.